## Annotation Analysis — `AnnotationAgent` spec & HITL metrics

What this notebook does:
- Shows class distribution and generated annotation spec
- Computes label quality metrics (Cohen's kappa) if HITL-corrected labels exist


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import yaml
from agents.annotation_agent import AnnotationAgent

sns.set_theme(style='whitegrid')

cfg = yaml.safe_load(Path('config.yaml').read_text(encoding='utf-8'))
ann_cfg = cfg.get('annotation', {})
label_space = ann_cfg.get('label_space', None)

agent = AnnotationAgent(modality=ann_cfg.get('modality', 'text'), label_space=label_space)

# Load latest finalized labeled dataset
runs = sorted(Path('artifacts').glob('run_*/data_labeled_final.csv'))
final_path = runs[-1] if runs else Path('data/labeled/data_labeled_final.csv')
df_final = pd.read_csv(final_path)
df_final.shape

In [ ]:
# Add label_auto so `check_quality()` can use the expected column
df_final['label_auto'] = df_final['label_final']
metrics = agent.check_quality(df_final)
metrics

In [ ]:
# Generated spec snapshot (from auto labels or fallback)
spec_path_candidates = sorted(Path('artifacts').glob('run_*/reports/annotation_spec.md'))
spec_path = spec_path_candidates[-1] if spec_path_candidates else Path('reports/annotation_spec.md')
spec_md = spec_path.read_text(encoding='utf-8') if spec_path.exists() else ''
spec_md[:1000]

In [ ]:
# Visualize label distribution
vc = df_final['label_final'].astype(str).value_counts()
plt.figure(figsize=(10,4))
sns.barplot(x=vc.index.astype(str), y=vc.values)
plt.xticks(rotation=45, ha='right')
plt.title('Final label distribution')
plt.tight_layout()
plt.show()


In [ ]:
# If HITL-corrected labels exist, compute kappa after human review
corrected_candidates = sorted(Path('artifacts').glob('run_*/review_queue_corrected.csv'))
corrected_path = corrected_candidates[-1] if corrected_candidates else None

if corrected_path and corrected_path.exists():
    corrected = pd.read_csv(corrected_path)
    if 'label_human' in corrected.columns:
        corrected['label_human'] = corrected['label_human'].astype('string')
        corrected = corrected[corrected['label_human'].notna() & (corrected['label_human'].str.len() > 0)].copy()
        df_metrics = df_final.merge(corrected[['text','label_human']], on='text', how='inner')
        df_metrics['label_auto'] = df_metrics['label_final']
        kappa_metrics = agent.check_quality(df_metrics)
        kappa_metrics
    else:
        'No label_human column found in corrected queue.'
else:
    'No corrected queue found; run HITL and resume the pipeline.'